# Replication: *Democratizing Optimization with Generative AI* (Simchi-Levi, Dai, Menache & Wu, SSRN 5511218)

Free, open models only: **Qwen 2.5 7B** and **Llama 3.1 8B** via Ollama. No API keys.

**Before running:** `Runtime → Change runtime type → T4 GPU → Save`, then `Runtime → Run all`. Nothing to edit.

## 1. Get the code

In [ ]:
%cd /content
REPO_URL = "https://github.com/Priyanshi8806/democratizing-optimization-replication.git"
!rm -rf /content/repo && git clone -q $REPO_URL /content/repo
%cd /content/repo
!ls

## 2. Install Python packages + run tests (expect `7 passed`)

In [ ]:
%cd /content/repo
!pip install -q -r requirements.txt
!python -m pytest -q tests/

## 3. Install Ollama and download Qwen + Llama (~10 min first time)

In [ ]:
%cd /content/repo
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time
subprocess.Popen("nohup ollama serve > /content/ollama.log 2>&1 &", shell=True)
time.sleep(5)
MODELS = ["qwen2.5:7b-instruct", "llama3.1:8b"]   # free open models, no API key
for m in MODELS:
    !ollama pull {m}
!ollama list

## 4. 4I demo (Insight, Interpretability, Interactivity, Improvisation)

In [ ]:
%cd /content/repo
from IPython.display import Markdown, display
for m in MODELS:
    !python scripts/demo.py --llm --model {m}
    tag = m.split(":")[0]
    !cp results/demo_transcript_llm.md results/demo_transcript_{tag}.md
    display(Markdown(f"### {m}\n" + open(f"results/demo_transcript_{tag}.md").read()[:4000]))

## 5. Parser benchmark: rules vs Qwen vs Llama

In [ ]:
%cd /content/repo
!python scripts/run_benchmark.py --backend rules
for m in MODELS:
    !python scripts/run_benchmark.py --backend llm --model {m}

In [ ]:
%cd /content/repo
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob("results/bench_*_summary.json")):
    s = json.load(open(f))
    for split in ("ID", "OOD"):
        rows.append({"backend": s["backend"], "split": split,
                     **{k: s[split][k] for k in ("n_questions", "question_acc", "set_acc", "e2e_acc", "median_parse_s")}})
pd.DataFrame(rows)

## 6. Figure 1 (Google Trends) — may be rate-limited from Colab

In [ ]:
%cd /content/repo
!python scripts/figure1_trends.py --fetch || echo "Trends blocked - use manual CSV (see scripts/figure1_trends.py)"
import os
from IPython.display import Image
if os.path.exists("results/figure1_trends.png"):
    display(Image("results/figure1_trends.png"))

## 6b. Multi-agent 4I system (Router + Insight / Interpretability / Interactivity / Improvisation agents)
Each 4I agent uses Planner → Executor (solver) → Validator → Explainer. The LLM only routes, parses and narrates; the solver decides; the Validator checks every plan and every explanation. Big changes need human approval.

In [ ]:
%cd /content/repo
from IPython.display import Markdown, display
for m in MODELS:
    !python scripts/multi_agent.py --llm --model {m} > /dev/null
    tag = m.split(":")[0]
    display(Markdown(open(f"results/multi_agent_{tag}.md").read()))

Try it yourself — type questions to the agents (stop the cell to exit):

In [ ]:
%cd /content/repo
!python scripts/multi_agent.py --llm --model qwen2.5:7b-instruct --chat

## 7. Download all results

In [ ]:
%cd /content/repo
!zip -qr /content/results.zip results data
from google.colab import files
files.download("/content/results.zip")